# RACE Dataset Exploratory Data Analysis

This notebook analyzes the RACE reading comprehension CSV files used by the Intelligent Reading Comprehension and Quiz Generation System.

In [ ]:
import os
from collections import Counter
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = os.path.join('..', 'data', 'raw')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

## 1. Dataset Overview

In [ ]:
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(name, df.shape)
    display(df.head(2))
    display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'nulls': df.isna().sum()}))

## 2. Article Analysis

In [ ]:
article_words = train_df['article'].fillna('').astype(str).str.split().map(len)
fig = px.histogram(article_words, nbins=60, title='Article Word Count Distribution', labels={'value': 'word count'})
fig.show()
print({'average': article_words.mean(), 'median': article_words.median(), 'min': article_words.min(), 'max': article_words.max()})
tokens = ' '.join(train_df['article'].fillna('').astype(str).str.lower()).split()
tokens = [t.strip('.,!?;:\"()[]') for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 2]
top20 = pd.DataFrame(Counter(tokens).most_common(20), columns=['word', 'count'])
px.bar(top20, x='word', y='count', title='Top 20 Article Words').show()

## 3. Question Analysis

In [ ]:
def qtype(q):
    first = str(q).lower().strip().split()[0] if str(q).strip() else 'other'
    return first.title() if first in {'what','who','where','when','why','how'} else 'Other'

qtypes = train_df['question'].map(qtype).value_counts().reset_index()
qtypes.columns = ['type', 'count']
display(qtypes)
px.bar(qtypes, x='type', y='count', title='Question Type Counts').show()
print('Average question length:', train_df['question'].fillna('').astype(str).str.split().map(len).mean())

## 4. Answer Balance

In [ ]:
answer_counts = train_df['answer'].value_counts().sort_index().reset_index()
answer_counts.columns = ['answer', 'count']
display(answer_counts)
px.bar(answer_counts, x='answer', y='count', title='Correct Answer Label Balance').show()

## 5. Option Length Analysis

In [ ]:
rows = []
for _, row in train_df.iterrows():
    for label in ['A','B','C','D']:
        rows.append({'is_correct': label == row['answer'], 'length': len(str(row[label]).split())})
option_lengths = pd.DataFrame(rows)
px.box(option_lengths, x='is_correct', y='length', title='Option Lengths: Correct vs Incorrect').show()
display(option_lengths.groupby('is_correct')['length'].describe())

## 6. TF-IDF Top Terms

In [ ]:
sample = train_df.sample(min(10000, len(train_df)), random_state=42)
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=2, max_features=5000)
X = tfidf.fit_transform(sample['article'].fillna('').astype(str))
terms = np.array(tfidf.get_feature_names_out())
for label in ['A','B','C','D']:
    label_scores = np.asarray(X[sample['answer'].values == label].mean(axis=0)).ravel()
    top_terms = terms[np.argsort(label_scores)[-20:][::-1]]
    print(label, ', '.join(top_terms))
wc = WordCloud(width=900, height=400, background_color='white').generate(' '.join(tokens[:50000]))
plt.figure(figsize=(12,5)); plt.imshow(wc); plt.axis('off'); plt.show()

## 7. Cosine Similarity Distribution

In [ ]:
sample = train_df.sample(min(1000, len(train_df)), random_state=42)
correct_sims, wrong_sims = [], []
vectorizer = TfidfVectorizer(stop_words='english', min_df=1)
for _, row in sample.iterrows():
    texts = [row['article'], row[row['answer']]] + [row[l] for l in ['A','B','C','D'] if l != row['answer']]
    mat = vectorizer.fit_transform([str(t) for t in texts])
    correct_sims.append(cosine_similarity(mat[0], mat[1]).ravel()[0])
    wrong_sims.extend(cosine_similarity(mat[0], mat[2:]).ravel().tolist())
sim_df = pd.DataFrame({'similarity': correct_sims + wrong_sims, 'type': ['correct']*len(correct_sims) + ['wrong']*len(wrong_sims)})
px.histogram(sim_df, x='similarity', color='type', barmode='overlay', nbins=50, title='Article-Option Cosine Similarity').show()

## 8. Key Findings Summary

- RACE passages are long enough that sparse lexical features are more memory-efficient than dense encodings.
- Correct answer labels should be checked for approximate A/B/C/D balance before training.
- Question types are usually dominated by What/Why/How patterns, so macro metrics matter.
- Correct options may have mild lexical similarity advantages, but distractors are often intentionally close.
- Option length analysis helps detect shortcut bias in traditional classifiers.